# Project 2: Graph RAG — Walkthrough

This notebook demonstrates how Graph RAG improves on standard vector-based RAG by incorporating
knowledge graph structure into the retrieval and generation process.

**Graph RAG vs Vector RAG:**
- **Vector RAG**: Embeds chunks, retrieves by similarity, loses structural context
- **Graph RAG**: Builds a knowledge graph, uses graph structure for context-aware retrieval
  - *Local search*: Start from an entity, traverse neighbors for focused answers
  - *Global search*: Use community summaries for broad, thematic questions

In [ ]:
# Setup and imports
import sys
import json
from pathlib import Path
from collections import Counter

from pydantic import BaseModel, Field

# Add projects dir to path for shared imports
sys.path.insert(0, str(Path(".").resolve().parent.parent))

from shared.llm_clients import (
    chat_completion,
    chat_completion_structured,
    get_embedding,
    get_embedding_batch,
)
from shared.document_loader import load_text_files

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path(".").resolve().parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Setup complete!")

In [ ]:
# Step 1: Load the corpus
corpus_dir = DATA_DIR / "corpus"
documents = load_text_files(str(corpus_dir))

print(f"Loaded {len(documents)} documents\n")
for doc in documents:
    print(f"  - {doc['filename']} ({len(doc['content'].split())} words)")

In [ ]:
# Step 2: Build a vector RAG baseline with ChromaDB
# We chunk documents and embed them for similarity search

def chunk_text(text, chunk_size=500, overlap=100):
    """Split text into overlapping chunks."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

# Chunk all documents
all_chunks = []
chunk_metadata = []
for doc in documents:
    chunks = chunk_text(doc["content"])
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({"source": doc["filename"], "chunk_idx": i})

print(f"Created {len(all_chunks)} chunks from {len(documents)} documents")

# Embed all chunks
print("Embedding chunks...")
chunk_embeddings = get_embedding_batch(all_chunks)
chunk_embeddings_np = np.array(chunk_embeddings)
print(f"Embeddings shape: {chunk_embeddings_np.shape}")

def vector_search(query, top_k=3):
    """Simple vector similarity search."""
    query_emb = np.array(get_embedding(query))
    similarities = chunk_embeddings_np @ query_emb / (
        np.linalg.norm(chunk_embeddings_np, axis=1) * np.linalg.norm(query_emb) + 1e-10
    )
    top_idx = np.argsort(similarities)[::-1][:top_k]
    return [(all_chunks[i], chunk_metadata[i], float(similarities[i])) for i in top_idx]

# Test vector search
results = vector_search("What are the main technologies discussed?")
print("\nVector search results:")
for chunk, meta, score in results:
    print(f"  [{score:.3f}] {meta['source']}:{meta['chunk_idx']} — {chunk[:100]}...")

In [ ]:
# Step 3: Build the knowledge graph (entity extraction + NetworkX)

class Entity(BaseModel):
    name: str = Field(description="Entity name")
    entity_type: str = Field(description="Entity type")

class Relationship(BaseModel):
    source: str = Field(description="Source entity")
    target: str = Field(description="Target entity")
    relation: str = Field(description="Relationship type")

class GraphExtraction(BaseModel):
    entities: list[Entity] = Field(description="Extracted entities")
    relationships: list[Relationship] = Field(description="Extracted relationships")

G = nx.DiGraph()

for doc in documents:
    print(f"Extracting from: {doc['filename']}")
    result = chat_completion_structured(
        prompt=f"Extract entities and relationships from this text:\n\n{doc['content'][:3000]}",
        output_schema=GraphExtraction,
        system="Extract entities and their relationships. Use concise, uppercase relationship types.",
    )
    for entity in result.entities:
        G.add_node(entity.name, entity_type=entity.entity_type)
    for rel in result.relationships:
        if rel.source in G and rel.target in G:
            G.add_edge(rel.source, rel.target, relation=rel.relation)

print(f"\nGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# Step 4: Community detection and visualization

%matplotlib inline

from networkx.algorithms.community import greedy_modularity_communities

# Detect communities
undirected = G.to_undirected()
communities = list(greedy_modularity_communities(undirected))
print(f"Found {len(communities)} communities\n")

# Assign community colors
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc948', '#b07aa1']
node_community = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_community[node] = i
    print(f"Community {i} ({len(comm)} members): {sorted(comm)[:5]}{'...' if len(comm) > 5 else ''}")

# Visualize
fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(G, k=2, seed=42)
node_colors = [colors[node_community.get(n, 0) % len(colors)] for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=500, alpha=0.8, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color='#cccccc', arrows=True, arrowsize=12, ax=ax)

ax.set_title("Knowledge Graph with Community Detection", fontsize=16)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Step 5: Local search — entity-focused retrieval
# Start from an entity, gather its neighborhood, and use as context for LLM

def local_search(G, entity_name, question, hops=2):
    """Answer a question using local graph context around an entity."""
    if entity_name not in G:
        return f"Entity '{entity_name}' not found in graph."
    
    # Gather neighborhood within N hops
    neighborhood = set()
    frontier = {entity_name}
    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            neighborhood.add(node)
            next_frontier |= set(G.successors(node)) | set(G.predecessors(node))
        frontier = next_frontier - neighborhood
    neighborhood |= frontier
    
    # Build context from local subgraph
    subgraph = G.subgraph(neighborhood)
    triples = []
    for s, t, d in subgraph.edges(data=True):
        triples.append(f"({s}) --[{d.get('relation', 'RELATED')}]--> ({t})")
    
    context = "\n".join(triples)
    
    answer = chat_completion(
        prompt=f"""Using this knowledge graph context centered on '{entity_name}':

{context}

Question: {question}""",
        system="Answer using only the provided graph context. Be specific.",
    )
    return answer

# Demo: pick an entity and ask about it
sample_entity = list(G.nodes())[0]
question = f"What is {sample_entity} connected to and how?"
print(f"Local search for: {sample_entity}")
print(f"Q: {question}")
print(f"A: {local_search(G, sample_entity, question)}")

In [ ]:
# Step 6: Global search — community-based retrieval
# Summarize each community, then use summaries to answer broad questions

community_summaries = []

for i, comm in enumerate(communities):
    # Get all triples involving community members
    subgraph = G.subgraph(comm)
    triples = [f"({s}) --[{d.get('relation', '')}]--> ({t})" for s, t, d in subgraph.edges(data=True)]
    members = sorted(comm)
    
    if not triples:
        summary = f"Community of isolated entities: {', '.join(members)}"
    else:
        summary = chat_completion(
            prompt=f"""Summarize this knowledge graph community in 2-3 sentences:
Members: {', '.join(members)}
Relationships:
{chr(10).join(triples)}""",
            system="Summarize the main theme and key relationships of this community.",
        )
    
    community_summaries.append({"id": i, "members": members, "summary": summary})
    print(f"Community {i}: {summary[:150]}...")
    print()

def global_search(question):
    """Answer broad questions using community summaries."""
    context = "\n\n".join(
        f"Community {cs['id']} ({', '.join(cs['members'][:5])}): {cs['summary']}"
        for cs in community_summaries
    )
    return chat_completion(
        prompt=f"""Using these knowledge graph community summaries:

{context}

Question: {question}""",
        system="Synthesize an answer from the community summaries. Provide a comprehensive overview.",
    )

# Demo
print("Global search demo:")
q = "What are the main themes and topics covered in this knowledge base?"
print(f"Q: {q}")
print(f"A: {global_search(q)}")

In [ ]:
# Step 7: Side-by-side comparison — Vector RAG vs Graph RAG

test_questions = [
    "What are the main technologies and how do they relate to each other?",
    "Give me an overview of the key themes in this corpus.",
]

for question in test_questions:
    print(f"{'='*60}")
    print(f"Q: {question}\n")
    
    # Vector RAG
    vector_results = vector_search(question, top_k=3)
    vector_context = "\n\n".join([chunk for chunk, _, _ in vector_results])
    vector_answer = chat_completion(
        prompt=f"Context:\n{vector_context}\n\nQuestion: {question}",
        system="Answer using only the provided context.",
    )
    print(f"[Vector RAG]: {vector_answer}\n")
    
    # Graph RAG (global)
    graph_answer = global_search(question)
    print(f"[Graph RAG]:  {graph_answer}\n")

## Analysis and Conclusions

### Key Differences

| Aspect | Vector RAG | Graph RAG |
|--------|------------|----------|
| **Retrieval** | Similarity-based chunks | Structure-aware traversal |
| **Context** | Flat text chunks | Entity relationships + community summaries |
| **Best for** | Specific factual questions | Relational and thematic questions |
| **Weakness** | Loses relationships between chunks | Higher setup cost (extraction) |

### When to Use Each

- **Vector RAG**: Simple Q&A, factoid retrieval, when relationships don't matter
- **Graph RAG Local**: Questions about specific entities and their connections
- **Graph RAG Global**: Broad thematic questions, summarization, multi-hop reasoning
- **Hybrid**: Combine both for the best of both worlds

### Next Steps

- **Project 3**: Build an agentic system that dynamically chooses between local/global/vector retrieval
- Try tuning community detection parameters (resolution, algorithm)
- Experiment with different chunk sizes and overlap for vector RAG